# TripPulse - Week 6
## Data Quality: Silver Candidate to Trusted Silver and Quarantine

**Project:** TripPulse — Urban Mobility Analytics  
**Week:** 6  
**Technology:** Databricks Free Edition | Spark SQL | Delta tables

> **Week 6 job:** Test every TripPulse Week-5 Candidate record against the approved TripPulse DQ rules. Records that pass become **Trusted Silver**. Records that fail move to **Quarantine with all applicable failure reasons retained**. No record is silently deleted.


## 1. Outcome first - what will TripPulse produce?

Week 5 prepared typed and standardised **Silver Candidate** tables. Candidate means *ready for quality assessment*; it does not yet mean trusted.

| Week-5 input | Week-6 Trusted output | Week-6 Quarantine output |
|---|---|---|
| `silver_zones_candidate` | `silver_trippulse_zones_trusted` | `quarantine_trippulse_zones` |
| `silver_drivers_candidate` | `silver_trippulse_drivers_trusted` | `quarantine_trippulse_drivers` |
| `silver_trips_candidate` | `silver_trippulse_trips_trusted` | `quarantine_trippulse_trips` |
| `silver_payments_candidate` | `silver_trippulse_payments_trusted` | `quarantine_trippulse_payments` |

A separate rule-detail table is also created for each entity so that **every failed rule** can be retained even when one physical row fails multiple rules.

The non-negotiable proof for every entity is:

`Candidate rows = Trusted rows + Quarantine rows`

**Success standard:** every Candidate physical record appears exactly once on one side of the split, and every quarantined row explains why it failed.


## 2. Week 6 in one minute

Think of an airport security queue:

| TripPulse concept | Airport analogy | Meaning |
|---|---|---|
| Silver Candidate | passengers waiting for checks | prepared records awaiting DQ |
| DQ rulebook | documented security checks | approved conditions, not guesses |
| Trusted Silver | cleared passengers | records that passed every governing rule |
| Quarantine | secondary inspection | records held with clear failure reasons |
| Diagnostic profile | an observation | useful to investigate, but not automatically a rejection |

> A value can be unusual without being invalid. Quarantine only when an approved TripPulse rule fails.


## 3. What you will learn and do

By the end of this notebook, you will be able to:

1. explain Candidate, Trusted Silver and Quarantine;
2. implement completeness, uniqueness, reference, domain, range, chronology and lineage checks;
3. use readable `CASE WHEN` statements to mark each rule `PASS` or `FAIL`;
4. keep every failure reason when one record breaks several rules;
5. route records into Trusted Silver or entity-specific Quarantine tables;
6. reconcile counts and physical-record membership;
7. explain controlled reruns and the correct-and-replay pattern;
8. adapt the method without copying unrelated project rules.

**Coding approach:** prepare -> check -> inspect -> summarise -> route -> prove.


## 4. Today's seven-action journey

| Action | What you do | Evidence produced |
|---:|---|---|
| 1 | Confirm the Week-5 Candidate handoff | tables, counts and lineage |
| 2 | Read the TripPulse DQ rulebook | rule IDs, meanings and severity |
| 3 | Apply one rule family at a time | visible `PASS/FAIL` columns |
| 4 | Capture all failures for each row | failure IDs and reasons |
| 5 | Route each entity | Trusted and Quarantine Delta tables |
| 6 | Prove no silent loss | count and membership reconciliation |
| 7 | Test rerun and explain replay | repeat-run evidence and recovery note |

Do not jump directly to table creation. The inspection steps are where the quality decision becomes explainable.


## 5. Three levels of quality checking

| Level | Question | TripPulse example | Changes route? |
|---|---|---|---|
| Profile | What values and patterns exist? | count trips by status | No |
| Diagnostic | Does something deserve investigation? | non-zero fare variance | No, until a threshold is approved |
| Governing rule | Does the record violate an approved condition? | impossible trip timestamp order | Yes |

**Why this matters:** unusual data is not automatically invalid. Only the approved TripPulse rulebook controls routing.


## 6. TripPulse DQ rulebook - written for humans

### 6.1 Zone rule

| Rule ID | Record fails when... | Why it matters | Severity |
|---|---|---|---|
| `DQ-ZON-001` | zone identity, category, city, active flag or effective date violates the approved contract | invalid zone master data can break downstream demand analysis | Critical |

### 6.2 Driver rule

| Rule ID | Record fails when... | Why it matters | Severity |
|---|---|---|---|
| `DQ-DRV-001` | driver identity, home-zone reference, dates, vehicle/service compatibility, status, rating, lifetime trips or source version is invalid | invalid driver master data can make driver and trip joins untrustworthy | Critical |

### 6.3 Trip rules

| Rule ID | Rule | Severity |
|---|---|---|
| `DQ-TRIP-001` | trip key completeness, format and uniqueness | Critical |
| `DQ-TRIP-002` | pickup/dropoff zones or applicable driver references do not resolve | Critical |
| `DQ-TRIP-003` | lifecycle timestamps are chronologically impossible | Critical |
| `DQ-TRIP-004` | trip status is inconsistent with timestamps, cancellation fields or driver presence | Major |
| `DQ-TRIP-005` | driver is inactive, missing when required, or incompatible with service | Major |
| `DQ-TRIP-006` | distance values violate approved ranges or lifecycle expectations | Major |
| `DQ-TRIP-007` | fare/surge values violate approved ranges or lifecycle expectations | Major |
| `DQ-TRIP-008` | request window or required Bronze lineage is invalid | Major |

### 6.4 Payment rules

| Rule ID | Rule | Severity |
|---|---|---|
| `DQ-PAY-001` | payment key/format/uniqueness or trusted-trip reference fails | Critical |
| `DQ-PAY-002` | attempt sequencing, final-attempt logic, payment fields, timing or successful-final reconciliation fails | Major |

Rule IDs are stable names used consistently in code, evidence and mentor discussions.


## 7. Scenario gallery - predict before running

| Scenario | Expected route | Reason |
|---|---|---|
| Valid zone row | Trusted | `DQ-ZON-001` passes |
| Duplicate driver ID | Quarantine | `DQ-DRV-001` |
| Trip with unknown pickup zone | Quarantine | `DQ-TRIP-002` |
| Return-like timestamp order in a trip | Quarantine | `DQ-TRIP-003` |
| Completed trip without a driver | Quarantine | `DQ-TRIP-004` |
| Driver service incompatible with trip service | Quarantine | `DQ-TRIP-005` |
| Negative/invalid distance | Quarantine | `DQ-TRIP-006` |
| Invalid fare or surge | Quarantine | `DQ-TRIP-007` |
| Trip outside declared batch window | Quarantine | `DQ-TRIP-008` |
| Payment referencing a non-trusted trip | Quarantine | `DQ-PAY-001` |
| One row breaks several rules | one Quarantine row with several rule IDs | evaluate every rule, not only the first |

Do not fabricate expected defect counts. The notebook should be executed in the team's Databricks workspace to obtain real results.


## 8. Confirm the Week-5 handoff

### 8.1 Select the working location

The supplied Week-5 TripPulse notebook uses the `TripPulse` catalog and `default` schema.


In [ ]:
%sql
USE CATALOG `TripPulse`;
USE SCHEMA `default`;

SELECT current_catalog() AS active_catalog,
       current_schema() AS active_schema;


### 8.2 Confirm all Candidate inputs

Week 6 reads the outputs of Week 5. It does not rebuild the Silver transformations.


In [ ]:
%sql
SHOW TABLES LIKE 'silver_*_candidate';


### 8.3 Record the starting counts

These are the control totals that must be explained at the end.


In [ ]:
%sql
SELECT 'zones' AS entity, COUNT(*) AS candidate_rows
FROM silver_zones_candidate
UNION ALL
SELECT 'drivers', COUNT(*) FROM silver_drivers_candidate
UNION ALL
SELECT 'trips', COUNT(*) FROM silver_trips_candidate
UNION ALL
SELECT 'payments', COUNT(*) FROM silver_payments_candidate
ORDER BY entity;


### 8.4 Confirm physical-record lineage fields

Week 5 retained `_source_file_name`, `_source_file_path`, `_ingested_at`, `_ingestion_run_id`, `_bronze_schema_version` and `_bronze_record_hash`.

`_bronze_record_hash` is the physical-row trace used to prove that a Candidate row was routed exactly once.


In [ ]:
%sql
SELECT trip_id,
       _source_file_name,
       _source_file_path,
       _ingestion_run_id,
       _bronze_record_hash,
       _bronze_schema_version
FROM silver_trips_candidate
LIMIT 5;


**Checkpoint:** explain the difference between `trip_id` (business identity) and `_bronze_record_hash` (physical-row traceability).


## 9. Learn the code pattern with one rule

The main pattern is:

```sql
CASE WHEN invalid_condition THEN 'FAIL' ELSE 'PASS' END
```

Start with the TripPulse zone rule. This cell only demonstrates the rule; it does not write a final table.


In [ ]:
%sql
SELECT zone_id,
       zone_name,
       city_code,
       zone_type,
       demand_band,
       CASE
         WHEN zone_id IS NULL OR trim(zone_id) = ''
           OR zone_id NOT RLIKE '^ZON-[0-9]{3}$'
           OR city_code <> 'TPC'
           OR zone_type NOT IN ('residential','commercial','transit_hub','education','mixed_use','airport')
           OR demand_band NOT IN ('low','medium','high')
           OR is_active IS NULL
           OR effective_from IS NULL
           OR effective_from < DATE '2025-01-01'
           OR effective_from > DATE '2026-03-31'
         THEN 'FAIL'
         ELSE 'PASS'
       END AS zon001_check
FROM silver_zones_candidate
LIMIT 20;


**Read it aloud:** “When the zone violates any approved zone reference condition, mark FAIL; otherwise mark PASS.”


## 10. Build Zone DQ in small steps

### 10.1 Find duplicate zone business keys

The approved TripPulse zone rule also treats repeated `(city_code, zone_name)` business identity as invalid.


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW duplicate_zone_ids AS
SELECT zone_id, COUNT(*) AS occurrences
FROM silver_zones_candidate
WHERE zone_id IS NOT NULL AND trim(zone_id) <> ''
GROUP BY zone_id
HAVING COUNT(*) > 1;

CREATE OR REPLACE TEMP VIEW duplicate_zone_business_keys AS
SELECT city_code, zone_name, COUNT(*) AS occurrences
FROM silver_zones_candidate
WHERE city_code IS NOT NULL
  AND zone_name IS NOT NULL
  AND trim(zone_name) <> ''
GROUP BY city_code, zone_name
HAVING COUNT(*) > 1;


### 10.2 Apply the zone rule

All zone conditions are represented in one governing rule because `DQ-ZON-001` is the approved rule ID.


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW zones_checked AS
SELECT z.*,
  CASE
    WHEN z.zone_id IS NULL OR trim(z.zone_id) = ''
      OR z.zone_id NOT RLIKE '^ZON-[0-9]{3}$'
      OR d.zone_id IS NOT NULL
      OR db.city_code IS NOT NULL
      OR z.zone_name IS NULL OR trim(z.zone_name) = ''
      OR z.city_code <> 'TPC'
      OR z.zone_type NOT IN ('residential','commercial','transit_hub','education','mixed_use','airport')
      OR z.demand_band NOT IN ('low','medium','high')
      OR z.is_active IS NULL
      OR z.effective_from IS NULL
      OR z.effective_from < DATE '2025-01-01'
      OR z.effective_from > DATE '2026-03-31'
    THEN 'FAIL' ELSE 'PASS'
  END AS dq_zon_001
FROM silver_zones_candidate z
LEFT JOIN duplicate_zone_ids d
  ON z.zone_id = d.zone_id
LEFT JOIN duplicate_zone_business_keys db
  ON z.city_code = db.city_code
 AND z.zone_name = db.zone_name;


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW zones_routed AS
SELECT *,
  CASE WHEN dq_zon_001 = 'FAIL' THEN 'DQ-ZON-001' ELSE '' END AS failed_rule_ids,
  CASE WHEN dq_zon_001 = 'FAIL' THEN 'ZONE_REFERENCE_INVALID' ELSE '' END AS failure_reason,
  CASE WHEN dq_zon_001 = 'FAIL' THEN 'Critical' ELSE 'NONE' END AS highest_severity,
  CASE WHEN dq_zon_001 = 'FAIL' THEN 'FAIL' ELSE 'PASS' END AS dq_status,
  current_timestamp() AS dq_checked_at,
  'TRIPPULSE-W06-V1' AS dq_ruleset_version
FROM zones_checked;


## 11. Build Driver DQ

Drivers are evaluated against the Candidate zone set first. Only drivers that pass the driver rule become Trusted Silver and are eligible for trip reference checks.


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW zones_trusted_keys AS
SELECT DISTINCT zone_id
FROM zones_routed
WHERE dq_status = 'PASS'
  AND zone_id IS NOT NULL;


### 11.1 Apply driver reference, domain, date and compatibility checks


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW drivers_checked AS
SELECT d.*,
  CASE
    WHEN d.driver_id IS NULL OR trim(d.driver_id) = ''
      OR d.driver_id NOT RLIKE '^DRV-[0-9]{6}$'
      OR dd.driver_id IS NOT NULL
      OR d.home_zone_id IS NULL
      OR z.zone_id IS NULL
      OR d.onboard_date IS NULL
      OR d.onboard_date < DATE '2023-01-01'
      OR d.onboard_date > DATE '2026-03-31'
      OR d.vehicle_type NOT IN ('bike','auto','mini','sedan')
      OR d.service_type NOT IN ('bike_taxi','auto','mini','sedan')
      OR (d.service_type = 'bike_taxi' AND d.vehicle_type <> 'bike')
      OR (d.service_type = 'auto' AND d.vehicle_type <> 'auto')
      OR (d.service_type = 'mini' AND d.vehicle_type <> 'mini')
      OR (d.service_type = 'sedan' AND d.vehicle_type <> 'sedan')
      OR d.driver_status NOT IN ('active','inactive','suspended')
      OR (d.rating IS NOT NULL AND (d.rating < 1 OR d.rating > 5))
      OR d.lifetime_completed_trips IS NULL
      OR d.lifetime_completed_trips < 0
      OR d.lifetime_completed_trips > 50000
      OR d.last_status_update_ts IS NULL
      OR d.last_status_update_ts < TIMESTAMP '2025-01-01 00:00:00'
      OR d.last_status_update_ts > TIMESTAMP '2026-04-01 01:00:00'
      OR d.source_record_version IS NULL
      OR d.source_record_version <= 0
    THEN 'FAIL' ELSE 'PASS'
  END AS dq_drv_001
FROM silver_drivers_candidate d
LEFT JOIN (
  SELECT driver_id, COUNT(*) AS occurrences
  FROM silver_drivers_candidate
  WHERE driver_id IS NOT NULL AND trim(driver_id) <> ''
  GROUP BY driver_id
  HAVING COUNT(*) > 1
) dd ON d.driver_id = dd.driver_id
LEFT JOIN zones_trusted_keys z
  ON d.home_zone_id = z.zone_id;


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW drivers_routed AS
SELECT *,
  CASE WHEN dq_drv_001 = 'FAIL' THEN 'DQ-DRV-001' ELSE '' END AS failed_rule_ids,
  CASE WHEN dq_drv_001 = 'FAIL' THEN 'DRIVER_REFERENCE_INVALID' ELSE '' END AS failure_reason,
  CASE WHEN dq_drv_001 = 'FAIL' THEN 'Critical' ELSE 'NONE' END AS highest_severity,
  CASE WHEN dq_drv_001 = 'FAIL' THEN 'FAIL' ELSE 'PASS' END AS dq_status,
  current_timestamp() AS dq_checked_at,
  'TRIPPULSE-W06-V1' AS dq_ruleset_version
FROM drivers_checked;


## 12. Build Trip DQ - complete worked example

Trips use eight approved governing rules. The rules are evaluated independently so one physical trip can retain several failure IDs.


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW duplicate_trip_ids AS
SELECT trip_id, COUNT(*) AS occurrences
FROM silver_trips_candidate
WHERE trip_id IS NOT NULL AND trim(trip_id) <> ''
GROUP BY trip_id
HAVING COUNT(*) > 1;


### 12.1 Prepare trusted reference keys


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW trusted_zone_keys AS
SELECT DISTINCT zone_id
FROM zones_routed
WHERE dq_status = 'PASS'
  AND zone_id IS NOT NULL;

CREATE OR REPLACE TEMP VIEW trusted_driver_reference AS
SELECT driver_id, service_type, driver_status
FROM drivers_routed
WHERE dq_status = 'PASS'
  AND driver_id IS NOT NULL;


### 12.2 Rule DQ-TRIP-001 - key completeness and uniqueness


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW trips_key_checked AS
SELECT t.*,
  CASE
    WHEN t.trip_id IS NULL OR trim(t.trip_id) = ''
      OR t.trip_id NOT RLIKE '^TRP-[0-9]{8}-[0-9]{6}$'
      OR d.trip_id IS NOT NULL
    THEN 'FAIL' ELSE 'PASS'
  END AS dq_trip_001
FROM silver_trips_candidate t
LEFT JOIN duplicate_trip_ids d
  ON t.trip_id = d.trip_id;


### 12.3 Rules DQ-TRIP-002 to DQ-TRIP-005 - references, chronology, status and compatibility


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW trips_core_checked AS
SELECT t.*,

  CASE
    WHEN pz.zone_id IS NULL OR dz.zone_id IS NULL
      OR (t.driver_id IS NOT NULL AND trim(t.driver_id) <> '' AND td.driver_id IS NULL)
      OR (t.trip_status IN ('completed','cancelled_by_driver')
          AND (t.driver_id IS NULL OR trim(t.driver_id) = ''))
    THEN 'FAIL' ELSE 'PASS'
  END AS dq_trip_002,

  CASE
    WHEN t.request_ts IS NULL THEN 'FAIL'
    WHEN t.driver_accept_ts IS NOT NULL AND t.driver_accept_ts < t.request_ts THEN 'FAIL'
    WHEN t.pickup_ts IS NOT NULL AND (t.driver_accept_ts IS NULL OR t.pickup_ts < t.driver_accept_ts) THEN 'FAIL'
    WHEN t.dropoff_ts IS NOT NULL AND (t.pickup_ts IS NULL OR t.dropoff_ts <= t.pickup_ts) THEN 'FAIL'
    WHEN t.cancel_ts IS NOT NULL AND t.cancel_ts < t.request_ts THEN 'FAIL'
    ELSE 'PASS'
  END AS dq_trip_003,

  CASE
    WHEN t.trip_status NOT IN ('completed','cancelled_by_rider','cancelled_by_driver','unfulfilled') THEN 'FAIL'
    WHEN t.trip_status = 'completed'
      AND (t.dropoff_ts IS NULL OR t.pickup_ts IS NULL OR t.driver_accept_ts IS NULL
           OR t.cancel_ts IS NOT NULL OR t.cancellation_reason IS NOT NULL
           OR t.driver_id IS NULL OR trim(t.driver_id) = '') THEN 'FAIL'
    WHEN t.trip_status = 'cancelled_by_rider'
      AND (t.cancel_ts IS NULL OR t.cancellation_reason IS NULL
           OR t.pickup_ts IS NOT NULL OR t.dropoff_ts IS NOT NULL) THEN 'FAIL'
    WHEN t.trip_status = 'cancelled_by_driver'
      AND (t.cancel_ts IS NULL OR t.cancellation_reason IS NULL
           OR t.driver_id IS NULL OR trim(t.driver_id) = ''
           OR t.pickup_ts IS NOT NULL OR t.dropoff_ts IS NOT NULL) THEN 'FAIL'
    WHEN t.trip_status = 'unfulfilled'
      AND (t.driver_id IS NOT NULL AND trim(t.driver_id) <> ''
           OR t.driver_accept_ts IS NOT NULL
           OR t.pickup_ts IS NOT NULL
           OR t.dropoff_ts IS NOT NULL
           OR t.cancel_ts IS NOT NULL
           OR t.cancellation_reason <> 'no_driver_found') THEN 'FAIL'
    WHEN t.cancellation_reason IS NOT NULL
      AND t.cancellation_reason NOT IN
          ('rider_changed_plan','driver_unavailable','no_driver_found',
           'excessive_wait','payment_issue','other') THEN 'FAIL'
    ELSE 'PASS'
  END AS dq_trip_004,

  CASE
    WHEN t.service_type NOT IN ('bike_taxi','auto','mini','sedan') THEN 'FAIL'
    WHEN t.driver_id IS NOT NULL AND trim(t.driver_id) <> ''
      AND (td.driver_id IS NULL OR td.driver_status <> 'active'
           OR td.service_type <> t.service_type) THEN 'FAIL'
    ELSE 'PASS'
  END AS dq_trip_005

FROM trips_key_checked t
LEFT JOIN trusted_zone_keys pz ON t.pickup_zone_id = pz.zone_id
LEFT JOIN trusted_zone_keys dz ON t.dropoff_zone_id = dz.zone_id
LEFT JOIN trusted_driver_reference td ON t.driver_id = td.driver_id;


### 12.4 Rules DQ-TRIP-006 to DQ-TRIP-008 - measures, fare/surge and window/lineage


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW trips_all_checked AS
SELECT t.*,

  CASE
    WHEN t.estimated_distance_km IS NULL
      OR t.estimated_distance_km < 0.3
      OR t.estimated_distance_km > 80.0 THEN 'FAIL'
    WHEN t.trip_status = 'completed'
      AND (t.actual_distance_km IS NULL
           OR t.actual_distance_km < 0.3
           OR t.actual_distance_km > 100.0) THEN 'FAIL'
    WHEN t.trip_status <> 'completed'
      AND t.actual_distance_km IS NOT NULL THEN 'FAIL'
    ELSE 'PASS'
  END AS dq_trip_006,

  CASE
    WHEN t.estimated_fare_inr IS NULL
      OR t.estimated_fare_inr < 20
      OR t.estimated_fare_inr > 5000 THEN 'FAIL'
    WHEN t.surge_multiplier IS NULL
      OR t.surge_multiplier < 1
      OR t.surge_multiplier > 3 THEN 'FAIL'
    WHEN t.trip_status = 'completed'
      AND (t.final_fare_inr IS NULL
           OR t.final_fare_inr < 20
           OR t.final_fare_inr > 6000) THEN 'FAIL'
    WHEN t.trip_status <> 'completed'
      AND t.final_fare_inr IS NOT NULL THEN 'FAIL'
    ELSE 'PASS'
  END AS dq_trip_007,

  CASE
    WHEN t.request_ts IS NULL
      OR t.request_ts < TIMESTAMP '2026-01-01 00:00:00'
      OR t.request_ts > TIMESTAMP '2026-03-31 23:59:59' THEN 'FAIL'
    WHEN t.record_created_ts IS NULL THEN 'FAIL'
    WHEN t.record_created_ts <
         COALESCE(t.dropoff_ts,t.cancel_ts,t.pickup_ts,t.driver_accept_ts,t.request_ts) THEN 'FAIL'
    WHEN t.record_created_ts > TIMESTAMP '2026-04-01 01:00:00' THEN 'FAIL'
    WHEN t._source_file_name IS NULL
      OR t._ingestion_run_id IS NULL
      OR t._bronze_record_hash IS NULL
      OR t._bronze_schema_version IS NULL THEN 'FAIL'
    ELSE 'PASS'
  END AS dq_trip_008

FROM trips_core_checked t;


### 12.5 Inspect all eight trip rule outcomes before routing


In [ ]:
%sql
SELECT trip_id,
       trip_status,
       dq_trip_001,
       dq_trip_002,
       dq_trip_003,
       dq_trip_004,
       dq_trip_005,
       dq_trip_006,
       dq_trip_007,
       dq_trip_008
FROM trips_all_checked
LIMIT 20;


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW trips_dq AS
SELECT *,
  concat_ws(', ',
    CASE WHEN dq_trip_001 = 'FAIL' THEN 'DQ-TRIP-001' END,
    CASE WHEN dq_trip_002 = 'FAIL' THEN 'DQ-TRIP-002' END,
    CASE WHEN dq_trip_003 = 'FAIL' THEN 'DQ-TRIP-003' END,
    CASE WHEN dq_trip_004 = 'FAIL' THEN 'DQ-TRIP-004' END,
    CASE WHEN dq_trip_005 = 'FAIL' THEN 'DQ-TRIP-005' END,
    CASE WHEN dq_trip_006 = 'FAIL' THEN 'DQ-TRIP-006' END,
    CASE WHEN dq_trip_007 = 'FAIL' THEN 'DQ-TRIP-007' END,
    CASE WHEN dq_trip_008 = 'FAIL' THEN 'DQ-TRIP-008' END
  ) AS failed_rule_ids
FROM trips_all_checked;


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW trips_routed AS
SELECT *,
  CASE WHEN failed_rule_ids = '' THEN 'PASS' ELSE 'FAIL' END AS dq_status,
  CASE
    WHEN dq_trip_001 = 'FAIL'
      OR dq_trip_002 = 'FAIL'
      OR dq_trip_003 = 'FAIL' THEN 'CRITICAL'
    WHEN dq_trip_004 = 'FAIL'
      OR dq_trip_005 = 'FAIL'
      OR dq_trip_006 = 'FAIL'
      OR dq_trip_007 = 'FAIL'
      OR dq_trip_008 = 'FAIL' THEN 'MAJOR'
    ELSE 'NONE'
  END AS highest_severity,
  current_timestamp() AS dq_checked_at,
  'TRIPPULSE-W06-V1' AS dq_ruleset_version
FROM trips_dq;


### 12.6 View genuine multi-rule failures


In [ ]:
%sql
SELECT trip_id,
       trip_status,
       failed_rule_ids,
       highest_severity
FROM trips_routed
WHERE failed_rule_ids LIKE '%,%'
LIMIT 20;


**Why can rule-failure totals exceed quarantined rows?** Because one quarantined physical trip can fail several governing rules.


## 13. Build Payment DQ

Payments are evaluated after Trusted Trips exist because `DQ-PAY-001` requires the payment to reference a trusted trip.


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW trusted_trip_reference AS
SELECT trip_id,
       final_fare_inr,
       request_ts,
       dropoff_ts
FROM trips_routed
WHERE dq_status = 'PASS'
  AND trip_id IS NOT NULL;


### 13.1 DQ-PAY-001 - payment key, uniqueness and trusted-trip integrity


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW payments_key_checked AS
SELECT p.*,
  CASE
    WHEN p.payment_id IS NULL OR trim(p.payment_id) = ''
      OR p.payment_id NOT RLIKE '^PAY-[0-9]{9}$'
      OR dp.payment_id IS NOT NULL
      OR p.trip_id IS NULL OR trim(p.trip_id) = ''
      OR t.trip_id IS NULL
    THEN 'FAIL' ELSE 'PASS'
  END AS dq_pay_001
FROM silver_payments_candidate p
LEFT JOIN (
  SELECT payment_id, COUNT(*) AS occurrences
  FROM silver_payments_candidate
  WHERE payment_id IS NOT NULL AND trim(payment_id) <> ''
  GROUP BY payment_id
  HAVING COUNT(*) > 1
) dp ON p.payment_id = dp.payment_id
LEFT JOIN trusted_trip_reference t
  ON p.trip_id = t.trip_id;


### 13.2 DQ-PAY-002 - attempt sequence, final attempt and reconciliation logic


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW payments_all_checked AS
SELECT p.*,

  CASE
    WHEN p.attempt_number IS NULL OR p.attempt_number < 1 OR p.attempt_number > 3 THEN 'FAIL'
    WHEN p.payment_method NOT IN ('upi','card','wallet','cash') THEN 'FAIL'
    WHEN p.payment_status NOT IN ('success','failed','pending','refunded') THEN 'FAIL'
    WHEN p.amount_inr IS NULL OR p.amount_inr < 0 OR p.amount_inr > 6000 THEN 'FAIL'
    WHEN p.payment_ts IS NULL THEN 'FAIL'
    WHEN p.payment_status = 'failed'
      AND p.failure_reason NOT IN ('bank_declined','timeout','insufficient_funds','technical_error') THEN 'FAIL'
    WHEN p.payment_status <> 'failed' AND p.failure_reason IS NOT NULL THEN 'FAIL'
    WHEN p.is_final_attempt IS NULL THEN 'FAIL'
    WHEN p.payment_reference IS NULL
      OR p.payment_reference NOT RLIKE '^TPREF-[A-F0-9]{8}$' THEN 'FAIL'
    WHEN dpr.payment_reference IS NOT NULL THEN 'FAIL'
    WHEN g.group_rows <> g.distinct_attempts
      OR g.min_attempt <> 1
      OR g.max_attempt <> g.group_rows
      OR g.final_count <> 1 THEN 'FAIL'
    WHEN p.is_final_attempt = TRUE AND p.attempt_number <> g.max_attempt THEN 'FAIL'
    WHEN g.success_count > 1 THEN 'FAIL'
    WHEN p.is_final_attempt = TRUE
      AND p.payment_status = 'success'
      AND (t.final_fare_inr IS NULL OR abs(p.amount_inr - t.final_fare_inr) > 0.01) THEN 'FAIL'
    WHEN p.payment_ts < t.request_ts THEN 'FAIL'
    WHEN t.dropoff_ts IS NOT NULL AND p.payment_ts < t.dropoff_ts THEN 'FAIL'
    ELSE 'PASS'
  END AS dq_pay_002

FROM payments_key_checked p
LEFT JOIN trusted_trip_reference t
  ON p.trip_id = t.trip_id
LEFT JOIN (
  SELECT payment_reference, COUNT(*) AS occurrences
  FROM silver_payments_candidate
  WHERE payment_reference IS NOT NULL
  GROUP BY payment_reference
  HAVING COUNT(*) > 1
) dpr ON p.payment_reference = dpr.payment_reference
LEFT JOIN (
  SELECT trip_id,
         COUNT(*) AS group_rows,
         COUNT(DISTINCT attempt_number) AS distinct_attempts,
         MIN(attempt_number) AS min_attempt,
         MAX(attempt_number) AS max_attempt,
         SUM(CASE WHEN is_final_attempt = TRUE THEN 1 ELSE 0 END) AS final_count,
         SUM(CASE WHEN payment_status = 'success' THEN 1 ELSE 0 END) AS success_count
  FROM silver_payments_candidate
  GROUP BY trip_id
) g ON p.trip_id = g.trip_id;


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW payments_dq AS
SELECT *,
  concat_ws(', ',
    CASE WHEN dq_pay_001 = 'FAIL' THEN 'DQ-PAY-001' END,
    CASE WHEN dq_pay_002 = 'FAIL' THEN 'DQ-PAY-002' END
  ) AS failed_rule_ids
FROM payments_all_checked;

CREATE OR REPLACE TEMP VIEW payments_routed AS
SELECT *,
  CASE WHEN failed_rule_ids = '' THEN 'PASS' ELSE 'FAIL' END AS dq_status,
  CASE
    WHEN dq_pay_001 = 'FAIL' THEN 'CRITICAL'
    WHEN dq_pay_002 = 'FAIL' THEN 'MAJOR'
    ELSE 'NONE'
  END AS highest_severity,
  current_timestamp() AS dq_checked_at,
  'TRIPPULSE-W06-V1' AS dq_ruleset_version
FROM payments_dq;


## 14. Cover the wider DQ families - without inventing rules

| DQ family | TripPulse treatment this week |
|---|---|
| Completeness | required keys, required references and lineage govern routing |
| Uniqueness | zone, driver, trip, payment and payment-reference uniqueness where approved |
| Reference integrity | driver home zone; trip zones/drivers; payment-to-trusted-trip |
| Domain validity | approved zone, service, vehicle, status, cancellation and payment values |
| Numeric validity | approved distance, fare, surge, rating and lifetime-trip ranges |
| Timestamp presence/order | trip lifecycle chronology and payment timing |
| Status-to-field consistency | completed/cancelled/unfulfilled lifecycle rules |
| Conditional completeness | fields required only for particular trip/payment states |
| Cross-field reconciliation | successful final payment against trusted final fare |
| Compatibility | driver vehicle/service and trip service/driver service |
| Reporting window | trip request timestamp must be inside the declared batch window |
| Lineage | Bronze file, ingestion run, hash and schema version are retained |
| Multi-rule capture | all failed TripPulse rules are listed for the row |
| Split reconciliation/rerun/replay | proved later in this notebook |

A diagnostic such as non-zero `fare_variance_inr` is not automatically a governing DQ failure unless the approved rulebook says so.


### 14.1 Profile controlled values


In [ ]:
%sql
SELECT 'zone_type' AS profile, zone_type AS value, COUNT(*) AS rows
FROM silver_zones_candidate
GROUP BY zone_type
UNION ALL
SELECT 'demand_band', demand_band, COUNT(*)
FROM silver_zones_candidate
GROUP BY demand_band
UNION ALL
SELECT 'trip_status', trip_status, COUNT(*)
FROM silver_trips_candidate
GROUP BY trip_status
UNION ALL
SELECT 'service_type', service_type, COUNT(*)
FROM silver_trips_candidate
GROUP BY service_type
UNION ALL
SELECT 'payment_status', payment_status, COUNT(*)
FROM silver_payments_candidate
GROUP BY payment_status
ORDER BY profile, value;


### 14.2 Profile trip lifecycle combinations


In [ ]:
%sql
SELECT trip_status,
       CASE WHEN driver_accept_ts IS NULL THEN 'accept missing' ELSE 'accept present' END AS accept_state,
       CASE WHEN pickup_ts IS NULL THEN 'pickup missing' ELSE 'pickup present' END AS pickup_state,
       CASE WHEN dropoff_ts IS NULL THEN 'dropoff missing' ELSE 'dropoff present' END AS dropoff_state,
       CASE WHEN cancel_ts IS NULL THEN 'cancel missing' ELSE 'cancel present' END AS cancel_state,
       COUNT(*) AS rows
FROM silver_trips_candidate
GROUP BY trip_status, accept_state, pickup_state, dropoff_state, cancel_state
ORDER BY trip_status, accept_state, pickup_state, dropoff_state, cancel_state;


### 14.3 Profile Week-5 derived variances

Week 5 calculated `distance_variance_km` and `fare_variance_inr`. They are useful diagnostics; this notebook only routes them when an approved Week-6 governing rule explicitly requires it.


In [ ]:
%sql
SELECT
  CASE WHEN distance_variance_km = 0 THEN 'matches'
       WHEN distance_variance_km IS NULL THEN 'not comparable'
       ELSE 'non-zero variance' END AS distance_diagnostic,
  COUNT(*) AS rows
FROM silver_trips_candidate
GROUP BY distance_diagnostic;

SELECT
  CASE WHEN fare_variance_inr = 0 THEN 'matches'
       WHEN fare_variance_inr IS NULL THEN 'not comparable'
       ELSE 'non-zero variance' END AS fare_diagnostic,
  COUNT(*) AS rows
FROM silver_trips_candidate
GROUP BY fare_diagnostic;


### 14.4 Profile the declared reporting window


In [ ]:
%sql
SELECT MIN(request_ts) AS earliest_request,
       MAX(request_ts) AS latest_request,
       SUM(CASE WHEN request_ts IS NULL THEN 1 ELSE 0 END) AS missing_request_rows
FROM silver_trips_candidate;


## 15. Summarise rule results

### 15.1 Entity routing totals


In [ ]:
%sql
SELECT 'zones' AS entity, dq_status, COUNT(*) AS rows
FROM zones_routed GROUP BY dq_status
UNION ALL
SELECT 'drivers', dq_status, COUNT(*) FROM drivers_routed GROUP BY dq_status
UNION ALL
SELECT 'trips', dq_status, COUNT(*) FROM trips_routed GROUP BY dq_status
UNION ALL
SELECT 'payments', dq_status, COUNT(*) FROM payments_routed GROUP BY dq_status
ORDER BY entity, dq_status;


### 15.2 Trip rule scorecard

One physical trip can fail several rules, so rule totals can exceed the final Quarantine row count.


In [ ]:
%sql
SELECT 'DQ-TRIP-001 key' AS rule,
       SUM(CASE WHEN dq_trip_001 = 'FAIL' THEN 1 ELSE 0 END) AS failed_rows
FROM trips_routed
UNION ALL SELECT 'DQ-TRIP-002 references',
       SUM(CASE WHEN dq_trip_002 = 'FAIL' THEN 1 ELSE 0 END) FROM trips_routed
UNION ALL SELECT 'DQ-TRIP-003 chronology',
       SUM(CASE WHEN dq_trip_003 = 'FAIL' THEN 1 ELSE 0 END) FROM trips_routed
UNION ALL SELECT 'DQ-TRIP-004 status',
       SUM(CASE WHEN dq_trip_004 = 'FAIL' THEN 1 ELSE 0 END) FROM trips_routed
UNION ALL SELECT 'DQ-TRIP-005 compatibility',
       SUM(CASE WHEN dq_trip_005 = 'FAIL' THEN 1 ELSE 0 END) FROM trips_routed
UNION ALL SELECT 'DQ-TRIP-006 distance',
       SUM(CASE WHEN dq_trip_006 = 'FAIL' THEN 1 ELSE 0 END) FROM trips_routed
UNION ALL SELECT 'DQ-TRIP-007 fare/surge',
       SUM(CASE WHEN dq_trip_007 = 'FAIL' THEN 1 ELSE 0 END) FROM trips_routed
UNION ALL SELECT 'DQ-TRIP-008 window/lineage',
       SUM(CASE WHEN dq_trip_008 = 'FAIL' THEN 1 ELSE 0 END) FROM trips_routed;


### 15.3 Payment rule scorecard


In [ ]:
%sql
SELECT 'DQ-PAY-001 key/trip' AS rule,
       SUM(CASE WHEN dq_pay_001 = 'FAIL' THEN 1 ELSE 0 END) AS failed_rows
FROM payments_routed
UNION ALL
SELECT 'DQ-PAY-002 attempt/reconciliation',
       SUM(CASE WHEN dq_pay_002 = 'FAIL' THEN 1 ELSE 0 END)
FROM payments_routed;


## 16. Write Trusted Silver and Quarantine tables

The outputs retain Candidate columns plus DQ status, failed rule IDs, severity and ruleset metadata.

### 16.1 Zone outputs


In [ ]:
%sql
CREATE OR REPLACE TABLE silver_trippulse_zones_trusted USING DELTA AS
SELECT * FROM zones_routed WHERE dq_status = 'PASS';

CREATE OR REPLACE TABLE quarantine_trippulse_zones USING DELTA AS
SELECT * FROM zones_routed WHERE dq_status = 'FAIL';


### 16.2 Driver outputs


In [ ]:
%sql
CREATE OR REPLACE TABLE silver_trippulse_drivers_trusted USING DELTA AS
SELECT * FROM drivers_routed WHERE dq_status = 'PASS';

CREATE OR REPLACE TABLE quarantine_trippulse_drivers USING DELTA AS
SELECT * FROM drivers_routed WHERE dq_status = 'FAIL';


### 16.3 Trip outputs


In [ ]:
%sql
CREATE OR REPLACE TABLE silver_trippulse_trips_trusted USING DELTA AS
SELECT * FROM trips_routed WHERE dq_status = 'PASS';

CREATE OR REPLACE TABLE quarantine_trippulse_trips USING DELTA AS
SELECT * FROM trips_routed WHERE dq_status = 'FAIL';


### 16.4 Payment outputs


In [ ]:
%sql
CREATE OR REPLACE TABLE silver_trippulse_payments_trusted USING DELTA AS
SELECT * FROM payments_routed WHERE dq_status = 'PASS';

CREATE OR REPLACE TABLE quarantine_trippulse_payments USING DELTA AS
SELECT * FROM payments_routed WHERE dq_status = 'FAIL';


## 17. Rule-detail evidence

The entity Quarantine tables contain one row per failed physical record. The rule-detail views below preserve **every** rule failure, which is important for multi-rule failures.


In [ ]:
%sql
CREATE OR REPLACE TABLE quarantine_trippulse_trip_rule_details USING DELTA AS
SELECT trip_id,
       _bronze_record_hash,
       _source_file_name,
       _ingestion_run_id,
       'DQ-TRIP-001' AS rule_id,
       'Trip key completeness and uniqueness' AS rule_name,
       'Critical' AS severity,
       'TRIP_KEY_INVALID' AS failure_reason
FROM trips_routed WHERE dq_trip_001 = 'FAIL'
UNION ALL
SELECT trip_id, _bronze_record_hash, _source_file_name, _ingestion_run_id,
       'DQ-TRIP-002', 'Trip reference integrity', 'Critical', 'TRIP_REFERENCE_ORPHAN'
FROM trips_routed WHERE dq_trip_002 = 'FAIL'
UNION ALL
SELECT trip_id, _bronze_record_hash, _source_file_name, _ingestion_run_id,
       'DQ-TRIP-003', 'Trip timestamp chronology', 'Critical', 'TRIP_TIMESTAMP_SEQUENCE_INVALID'
FROM trips_routed WHERE dq_trip_003 = 'FAIL'
UNION ALL
SELECT trip_id, _bronze_record_hash, _source_file_name, _ingestion_run_id,
       'DQ-TRIP-004', 'Trip status consistency', 'Major', 'TRIP_STATUS_CONDITION_INVALID'
FROM trips_routed WHERE dq_trip_004 = 'FAIL'
UNION ALL
SELECT trip_id, _bronze_record_hash, _source_file_name, _ingestion_run_id,
       'DQ-TRIP-005', 'Service and driver compatibility', 'Major', 'TRIP_SERVICE_ASSIGNMENT_INVALID'
FROM trips_routed WHERE dq_trip_005 = 'FAIL'
UNION ALL
SELECT trip_id, _bronze_record_hash, _source_file_name, _ingestion_run_id,
       'DQ-TRIP-006', 'Distance validity', 'Major', 'TRIP_DISTANCE_INVALID'
FROM trips_routed WHERE dq_trip_006 = 'FAIL'
UNION ALL
SELECT trip_id, _bronze_record_hash, _source_file_name, _ingestion_run_id,
       'DQ-TRIP-007', 'Fare and surge validity', 'Major', 'TRIP_FARE_SURGE_INVALID'
FROM trips_routed WHERE dq_trip_007 = 'FAIL'
UNION ALL
SELECT trip_id, _bronze_record_hash, _source_file_name, _ingestion_run_id,
       'DQ-TRIP-008', 'Declared window and lineage', 'Major', 'TRIP_WINDOW_OR_LINEAGE_INVALID'
FROM trips_routed WHERE dq_trip_008 = 'FAIL';


In [ ]:
%sql
CREATE OR REPLACE TABLE quarantine_trippulse_payment_rule_details USING DELTA AS
SELECT payment_id,
       _bronze_record_hash,
       _source_file_name,
       _ingestion_run_id,
       'DQ-PAY-001' AS rule_id,
       'Payment key and trip integrity' AS rule_name,
       'Critical' AS severity,
       'PAYMENT_KEY_OR_TRIP_INVALID' AS failure_reason
FROM payments_routed WHERE dq_pay_001 = 'FAIL'
UNION ALL
SELECT payment_id, _bronze_record_hash, _source_file_name, _ingestion_run_id,
       'DQ-PAY-002', 'Payment attempt and reconciliation logic', 'Major', 'PAYMENT_LOGIC_INVALID'
FROM payments_routed WHERE dq_pay_002 = 'FAIL';


## 18. Inspect useful evidence - without exposing unnecessary data

### 18.1 Trusted trip example


In [ ]:
%sql
SELECT trip_id,
       trip_status,
       service_type,
       response_seconds,
       wait_seconds,
       trip_duration_seconds,
       is_completed,
       dq_status,
       dq_ruleset_version
FROM silver_trippulse_trips_trusted
LIMIT 10;


### 18.2 Quarantine trip example


In [ ]:
%sql
SELECT trip_id,
       trip_status,
       service_type,
       failed_rule_ids,
       highest_severity,
       dq_status
FROM quarantine_trippulse_trips
LIMIT 10;


### 18.3 Quarantine payment example


In [ ]:
%sql
SELECT payment_id,
       trip_id,
       attempt_number,
       payment_status,
       amount_inr,
       failed_rule_ids,
       highest_severity
FROM quarantine_trippulse_payments
LIMIT 10;


**Evidence rule:** screenshots support the working notebook; they do not replace code, counts, reconciliation or commit history. Do not fabricate output values.


## 19. Prove no silent loss

For each entity:

`Candidate rows = Trusted rows + Quarantine rows`

### 19.1 Count reconciliation


In [ ]:
%sql
SELECT 'zones' AS entity,
  (SELECT COUNT(*) FROM silver_zones_candidate) AS candidate,
  (SELECT COUNT(*) FROM silver_trippulse_zones_trusted) AS trusted,
  (SELECT COUNT(*) FROM quarantine_trippulse_zones) AS quarantine
UNION ALL
SELECT 'drivers',
  (SELECT COUNT(*) FROM silver_drivers_candidate),
  (SELECT COUNT(*) FROM silver_trippulse_drivers_trusted),
  (SELECT COUNT(*) FROM quarantine_trippulse_drivers)
UNION ALL
SELECT 'trips',
  (SELECT COUNT(*) FROM silver_trips_candidate),
  (SELECT COUNT(*) FROM silver_trippulse_trips_trusted),
  (SELECT COUNT(*) FROM quarantine_trippulse_trips)
UNION ALL
SELECT 'payments',
  (SELECT COUNT(*) FROM silver_payments_candidate),
  (SELECT COUNT(*) FROM silver_trippulse_payments_trusted),
  (SELECT COUNT(*) FROM quarantine_trippulse_payments);


**Pass condition:** for every entity, `candidate = trusted + quarantine`.

Count equality is necessary, but it does not alone prove the same physical records were routed.


### 19.2 Physical-record membership proof - trips


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW trip_route_membership AS
SELECT _bronze_record_hash, COUNT(*) AS route_occurrences
FROM (
  SELECT _bronze_record_hash FROM silver_trippulse_trips_trusted
  UNION ALL
  SELECT _bronze_record_hash FROM quarantine_trippulse_trips
)
GROUP BY _bronze_record_hash;

SELECT COUNT(*) AS candidate_trip_hashes_not_routed_once
FROM silver_trips_candidate c
LEFT JOIN trip_route_membership r
  ON c._bronze_record_hash = r._bronze_record_hash
WHERE r.route_occurrences IS NULL OR r.route_occurrences <> 1;


### 19.3 Physical-record membership proof - all entities


In [ ]:
%sql
SELECT 'zones' AS entity, COUNT(*) AS bad_membership
FROM (
  SELECT c._bronze_record_hash
  FROM silver_zones_candidate c
  LEFT JOIN (
    SELECT _bronze_record_hash FROM silver_trippulse_zones_trusted
    UNION ALL
    SELECT _bronze_record_hash FROM quarantine_trippulse_zones
  ) r ON c._bronze_record_hash = r._bronze_record_hash
  GROUP BY c._bronze_record_hash
  HAVING COUNT(r._bronze_record_hash) <> 1
)
UNION ALL
SELECT 'drivers', COUNT(*)
FROM (
  SELECT c._bronze_record_hash
  FROM silver_drivers_candidate c
  LEFT JOIN (
    SELECT _bronze_record_hash FROM silver_trippulse_drivers_trusted
    UNION ALL
    SELECT _bronze_record_hash FROM quarantine_trippulse_drivers
  ) r ON c._bronze_record_hash = r._bronze_record_hash
  GROUP BY c._bronze_record_hash
  HAVING COUNT(r._bronze_record_hash) <> 1
)
UNION ALL
SELECT 'trips', COUNT(*)
FROM (
  SELECT c._bronze_record_hash
  FROM silver_trips_candidate c
  LEFT JOIN (
    SELECT _bronze_record_hash FROM silver_trippulse_trips_trusted
    UNION ALL
    SELECT _bronze_record_hash FROM quarantine_trippulse_trips
  ) r ON c._bronze_record_hash = r._bronze_record_hash
  GROUP BY c._bronze_record_hash
  HAVING COUNT(r._bronze_record_hash) <> 1
)
UNION ALL
SELECT 'payments', COUNT(*)
FROM (
  SELECT c._bronze_record_hash
  FROM silver_payments_candidate c
  LEFT JOIN (
    SELECT _bronze_record_hash FROM silver_trippulse_payments_trusted
    UNION ALL
    SELECT _bronze_record_hash FROM quarantine_trippulse_payments
  ) r ON c._bronze_record_hash = r._bronze_record_hash
  GROUP BY c._bronze_record_hash
  HAVING COUNT(r._bronze_record_hash) <> 1
);


**Expected result:** zero for every entity. If not, stop and investigate missing lineage, row multiplication, or incorrect routing.


## 20. Controlled rerun test

1. Save the current reconciliation results.
2. Rerun the helper views, checked views and routed views in order.
3. Rerun the Trusted/Quarantine table-write cells.
4. Run count and physical-membership reconciliation again.

**Pass condition:** the same Candidate snapshot produces the same Trusted and Quarantine membership. `dq_checked_at` may change because the check execution time changes.

Do not demonstrate rerun safety by deleting outputs manually.


## 21. Correction and replay - the safe pattern

Suppose a quarantined TripPulse trip has an invalid service/driver assignment.

### Wrong approach

- edit a source file secretly;
- update the Quarantine table directly;
- insert the row straight into Trusted Silver;
- delete the failed row to improve the counts.

### Correct approach

1. confirm the authoritative correction and approval;
2. correct the governed upstream input;
3. rerun Bronze ingestion with lineage;
4. rerun Week-5 Candidate transformation;
5. apply **all** Week-6 TripPulse DQ rules again;
6. reconcile Candidate, Trusted and Quarantine;
7. record before/after evidence and the code/documentation change.

> A correction does not “promote” a quarantine row. It creates a newly evaluated pipeline result.


## 22. Common failures and recovery

| Problem | Why it happens | Recovery |
|---|---|---|
| Candidate table not found | Week 5 incomplete or wrong catalog/schema | return to Week 5; do not recreate Candidate here |
| join increases row count | duplicate reference keys | profile reference keys and repair the earliest issue |
| only first failure retained | rules were written as one `ELSE IF` chain | evaluate each rule independently and combine IDs |
| null comparison unexpectedly passes | SQL null logic ignored | explicitly test `IS NULL` |
| duplicate business IDs are silently deduplicated | code arbitrarily selected a survivor | quarantine all repeated physical rows unless an approved survivor rule exists |
| unusual variance is quarantined | diagnostic mistaken for governing rule | keep it as profiling unless approved |
| Candidate does not equal split | filter/join/route logic lost or multiplied rows | trace `_bronze_record_hash` |
| direct Quarantine update | pipeline governance bypassed | correct upstream and replay |
| real outputs copied into a reference notebook | evidence was fabricated/leaked | execute in Databricks and capture actual results |


## 23. Adaptation notes for TripPulse

This notebook follows the same Week-6 teaching method as the supplied PageLoop example, but its entities, columns and rules are TripPulse-specific.

| PageLoop method | TripPulse implementation |
|---|---|
| Branch master DQ | Zone DQ |
| Book master DQ | Driver DQ |
| Loan DQ | Trip DQ |
| Payment-style transaction DQ | Payment DQ |
| Candidate -> Trusted/Quarantine | Candidate -> TripPulse Trusted/Quarantine |
| physical hash membership | `_bronze_record_hash` membership |
| rule IDs | `DQ-ZON-*`, `DQ-DRV-*`, `DQ-TRIP-*`, `DQ-PAY-*` |

Do not copy PageLoop rule IDs or business conditions into TripPulse.


## 24. Team-of-three ownership

| Student | Deep ownership | Must explain in mentor review |
|---|---|---|
| Student A | rulebook, zone/driver completeness, uniqueness and references | why master-data rules are approved |
| Student B | trip chronology/status/range/compatibility and multi-rule capture | how one trip retains every failure reason |
| Student C | payment logic, reconciliation, rerun/replay and evidence | how the team proves no silent loss |

All team members should run the full notebook and understand the complete pipeline.


## 25. GitHub evidence and weekly log

### Required evidence

- TripPulse DQ rule catalogue with rule ID, condition, severity and failure reason;
- working checks for every approved rule;
- entity routing scorecards;
- Trusted and Quarantine Delta tables and schemas;
- Candidate/Trusted/Quarantine count reconciliation;
- physical-record membership proof;
- multi-rule failure evidence;
- rerun evidence;
- one documented correction-and-replay scenario;
- team contribution record.

### Suggested commits

- `w06: add TripPulse DQ rule checks and failure reasons`
- `w06: route TripPulse trusted and quarantine records`
- `w06: add reconciliation rerun replay evidence`

### AI Transparency Note

Record what AI assisted with, what was independently verified in Databricks, what was changed or rejected, and which student approved the final logic. Never claim a query is validated until the team executes it.


## 26. Exit checklist

- [ ] I can explain Candidate, Trusted Silver and Quarantine in my own words.
- [ ] All four Week-5 TripPulse Candidate tables exist.
- [ ] Every approved rule has a clear ID, condition, reason and severity.
- [ ] Zone, driver, trip and payment DQ checks are implemented.
- [ ] Every governing rule is visible as a `PASS/FAIL` result.
- [ ] Multi-rule failures retain every applicable rule ID.
- [ ] Eight TripPulse output tables were created: four Trusted + four Quarantine.
- [ ] Candidate = Trusted + Quarantine passes for every entity.
- [ ] Physical-record membership passes for every entity.
- [ ] Controlled rerun evidence is recorded.
- [ ] Correction and replay is documented without direct Quarantine-to-Trusted promotion.
- [ ] GitHub commits, Week Log, AI note and team contributions are current.

**Week-6 completion standard:** Gold may read only from governed Trusted Silver after all exit checks pass.


## 27. Viva and mentor-review questions

1. Why is Silver Candidate not yet Trusted Silver?
2. Why are duplicate Trip IDs not resolved by keeping the first row?
3. Why must trip references be checked against trusted zones/drivers?
4. Why is trip chronology null-aware?
5. How can one TripPulse trip contain several failed rule IDs?
6. Why can rule-failure totals exceed Quarantine row counts?
7. Why is a diagnostic variance not automatically a DQ failure?
8. What does Candidate = Trusted + Quarantine prove?
9. Why retain `_bronze_record_hash`?
10. Why must a corrected record replay every Week-6 rule?
11. Why are payments evaluated after Trusted Trips?
12. What evidence proves the Week-6 run is repeatable?

### Final boundary

**Completed here:** TripPulse batch DQ rules, failure context, severity, Trusted/Quarantine routing, reconciliation, rerun and replay guidance.

**Next:** governed Gold design and analytical outputs only after Trusted Silver is accepted.

> Build. Prove. Present. A trusted table is not trusted because of its name; it is trusted because its rules, evidence and reconciliation are explainable.
